# 04: Scorecard model and evaluation (corrected version)

Changes from the previous version:
1. Uses the train/test split written by notebook 03 (WOE fitted on train only).
2. Compares model grade with LendingClub grade **on the out-of-time test set only**, with loan counts per cell.
3. Adds the question that matters: **does the model add discrimination beyond LendingClub's own grade?** (AUC of the grade alone vs grade + model score).
4. Relative paths: run from the `Python/` folder.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd().parent if Path.cwd().name == 'Python' else Path.cwd()
DATA, DOCS = ROOT / 'Dataset', ROOT / 'Documentation'

woe_df = pd.read_csv(DATA / 'loans_woe_transformed.csv')
meta = pd.read_csv(DATA / 'loans_model_ready.csv', usecols=['id', 'grade', 'sub_grade', 'vintage_month'])
data = woe_df.merge(meta, on='id', how='left')

FEATURE_COLS = [c for c in woe_df.columns if c.endswith('_woe')]
print('Features used:', FEATURE_COLS)
print(data.shape)

In [ ]:
train = data[data['split'] == 'train'].copy()
test = data[data['split'] == 'test'].copy()
train_vintages, test_vintages = set(train['vintage_month']), set(test['vintage_month'])

print(f"Train: {len(train):,} loans, vintages {min(train_vintages)} to {max(train_vintages)}")
print(f"Test:  {len(test):,} loans, vintages {min(test_vintages)} to {max(test_vintages)}")
print(f"Train default rate: {train['defaulted'].mean():.2%}  |  Test default rate: {test['defaulted'].mean():.2%}")
print("Note: the test loans are the most recent vintages, so some defaults have not happened yet; "
      "the lower test default rate is partly that effect.")

In [ ]:
X_train, y_train = train[FEATURE_COLS], train['defaulted']
X_test, y_test = test[FEATURE_COLS], test['defaulted']

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

coef_table = pd.DataFrame({'feature': FEATURE_COLS, 'coefficient': model.coef_[0]}).sort_values(
    'coefficient', key=abs, ascending=False)
coef_table

In [ ]:
test_probs_default = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_probs_default)
gini = 2 * auc - 1
fpr, tpr, thresholds = roc_curve(y_test, test_probs_default)
ks = np.max(tpr - fpr) * 100

print(f"AUC:  {auc:.4f}")
print(f"Gini: {gini:.4f}")
print(f"KS:   {ks:.2f}")
print("\nRule-of-thumb bands: KS < 20 Poor | 20-40 Acceptable | 40-60 Good | 60-75 Very strong | > 75 check leakage")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(fpr, tpr, label=f'AUC = {auc:.3f}', color='steelblue')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (out-of-time test set)')
axes[0].legend()

ks_idx = np.argmax(tpr - fpr)
axes[1].plot(thresholds, tpr, label='Cumulative % bad (defaults) captured', color='firebrick')
axes[1].plot(thresholds, fpr, label='Cumulative % good captured', color='seagreen')
axes[1].axvline(thresholds[ks_idx], linestyle='--', color='gray')
axes[1].set_xlabel('Score threshold (predicted probability)')
axes[1].set_title(f'KS Curve: KS = {ks:.1f} at threshold {thresholds[ks_idx]:.3f}')
axes[1].invert_xaxis()
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
BASE_SCORE = 600
BASE_ODDS = 50       # at the base score, odds of 'good' to 'bad' are 50:1
PDO = 20             # every 20 points doubles the odds

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def prob_default_to_score(p_default, floor=300, ceiling=850):
    p_default = np.clip(p_default, 1e-6, 1 - 1e-6)
    odds_good = (1 - p_default) / p_default
    score = offset + factor * np.log(odds_good)
    return np.clip(score, floor, ceiling).round().astype(int)

test = test.copy()
test['model_score'] = prob_default_to_score(test_probs_default)

# Grade cut-offs come from the TRAIN score distribution, so they do not shift when new data arrives.
train_probs_default = model.predict_proba(X_train)[:, 1]
train_scores = prob_default_to_score(train_probs_default)
grade_edges = np.percentile(train_scores, [0, 10, 25, 45, 65, 80, 92, 100])
grade_labels = ['G', 'F', 'E', 'D', 'C', 'B', 'A']

test['model_risk_grade'] = pd.cut(test['model_score'], bins=grade_edges, labels=grade_labels, include_lowest=True)

grade_check = test.groupby('model_risk_grade', observed=True).agg(
    loans=('id', 'count'), avg_score=('model_score', 'mean'), default_rate=('defaulted', 'mean')
).sort_values('avg_score')
grade_check

In [ ]:
def calculate_psi(expected, actual, n_bins=10):
    """PSI between two score distributions. 'expected' = baseline (train), 'actual' = new (test)."""
    edges = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    expected_dist = np.histogram(expected, bins=edges)[0] / len(expected)
    actual_dist = np.histogram(actual, bins=edges)[0] / len(actual)
    expected_dist = np.clip(expected_dist, 1e-4, None)
    actual_dist = np.clip(actual_dist, 1e-4, None)
    return np.sum((actual_dist - expected_dist) * np.log(actual_dist / expected_dist))

psi_value = calculate_psi(train_scores, test['model_score'].values)
print(f"PSI (train vs. out-of-time test): {psi_value:.4f}")
print("< 0.1 no significant shift | 0.1-0.25 moderate shift | > 0.25 major shift, retrain")

In [ ]:
# Out-of-time test set only: loans per cell (LendingClub grade x model grade)
counts = pd.crosstab(test['grade'], test['model_risk_grade']).reindex(columns=list('ABCDEFG'))
counts

In [ ]:
# Default rate per cell, showing only cells with at least 100 loans (smaller cells are noise)
MIN_CELL = 100
cell = test.groupby(['grade', 'model_risk_grade'], observed=True).agg(
    loans=('id', 'count'), default_rate=('defaulted', 'mean')).reset_index()
rate = cell.pivot(index='grade', columns='model_risk_grade', values='default_rate').reindex(columns=list('ABCDEFG'))
n_loans = cell.pivot(index='grade', columns='model_risk_grade', values='loans').reindex(columns=list('ABCDEFG'))
(rate.where(n_loans >= MIN_CELL) * 100).round(1)

In [ ]:
# Within each LendingClub grade: how well does the model score rank defaults? (out-of-time)
rows = []
for g, x in test.groupby('grade'):
    if x['defaulted'].nunique() == 2:
        rows.append({'lc_grade': g, 'loans': len(x), 'defaults': int(x['defaulted'].sum()),
                     'model_auc_within_grade': roc_auc_score(x['defaulted'], -x['model_score'])})
within_grade = pd.DataFrame(rows)
within_grade

In [ ]:
# Does the model add anything beyond LendingClub's own grade? (fit on train, scored on out-of-time test)
grade_rank = {g: i for i, g in enumerate('ABCDEFG')}
sub_rank = {s: i for i, s in enumerate(sorted(data['sub_grade'].unique()))}
for part in (train, test):
    part['grade_rank'] = part['grade'].map(grade_rank)
    part['sub_rank'] = part['sub_grade'].map(sub_rank)
    part['model_logit'] = model.decision_function(part[FEATURE_COLS])

results = {'model_alone': roc_auc_score(test['defaulted'], test['model_logit'])}
for col, label in [('grade_rank', 'lc_grade'), ('sub_rank', 'lc_sub_grade')]:
    base = LogisticRegression(max_iter=1000).fit(train[[col]], train['defaulted'])
    both = LogisticRegression(max_iter=1000).fit(train[[col, 'model_logit']], train['defaulted'])
    results[f'{label}_alone'] = roc_auc_score(test['defaulted'], base.predict_proba(test[[col]])[:, 1])
    results[f'{label}_plus_model'] = roc_auc_score(test['defaulted'], both.predict_proba(test[[col, 'model_logit']])[:, 1])
    results[f'{label}_gain'] = results[f'{label}_plus_model'] - results[f'{label}_alone']

pd.Series(results).round(4)

In [ ]:
all_probs_default = model.predict_proba(data[FEATURE_COLS])[:, 1]
data['model_score'] = prob_default_to_score(all_probs_default)
data['model_risk_grade'] = pd.cut(data['model_score'], bins=grade_edges, labels=grade_labels, include_lowest=True)

output_cols = ['id', 'model_score', 'model_risk_grade']
data[output_cols].to_csv(DATA / 'loans_scored.csv', index=False)

eval_summary = {
    'features_used': FEATURE_COLS,
    'auc': round(float(auc), 4),
    'gini': round(float(gini), 4),
    'ks': round(float(ks), 2),
    'psi_train_vs_test': round(float(psi_value), 4),
    'train_vintages': [min(train_vintages), max(train_vintages)],
    'test_vintages': [min(test_vintages), max(test_vintages)],
    'train_default_rate': round(float(train['defaulted'].mean()), 4),
    'test_default_rate': round(float(test['defaulted'].mean()), 4),
    'test_loans': int(len(test)),
    'woe_fitted_on': 'training vintages only',
    'auc_comparison_out_of_time': {k: round(float(v), 4) for k, v in results.items()},
}
with open(DOCS / 'model_evaluation_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print('Saved', (DATA / 'loans_scored.csv').relative_to(ROOT), data[output_cols].shape)
print('Saved', (DOCS / 'model_evaluation_summary.json').relative_to(ROOT))
print(eval_summary)